# Lab 8: Variability, Correlation, and the Regression Line
*Elements of Data Science* <br>
Welcome to Lab 8! Fitting a line to data is one of the most important tools in scientific data analysis. In this lab you will **build that tool yourself**, one idea at a time, instead of asking a black-box function for an answer:

1. **Spread** — how much do sample means vary? The **standard deviation (SD)** becomes our ruler.
2. **Standard units** — measure every value with that ruler.
3. **Look** at a scatter plot and **guess** the relationship.
4. **Measure** the relationship with the correlation coefficient *r*.
5. **Build** the regression line from *r* — and check it against your own hand-drawn line.
6. **Apply** it to chemistry: how does an alcohol's size and shape affect its boiling point?
7. **Verify** your line with `np.polyfit`, now that you know what it does.

This material is covered in [Chapter 14.5](https://inferentialthinking.com/chapters/14/5/Variability_of_the_Sample_Mean.html) and [Chapter 15](https://inferentialthinking.com/chapters/15/Prediction.html) of the online <i>Inferential Thinking</i> textbook.

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Variability of the sample mean|SD of sample means $= \dfrac{\text{population SD}{\sqrt{\text{sample size}}$|
|Standard units|Measure values in SDs from the mean: mean 0, SD 1|
|Scatter plots|Describe direction, form, and strength of an association|
|Correlation *r*|The average product of $x$ and $y$ in standard units|
|Regression line|Slope $= r \cdot \dfrac{SD_y}{SD_x}$, passing through $(\bar{x}, \bar{y})$|
|Least squares|The regression line has the smallest sum of squared errors|
|Molecular visualization|Draw 2D and 3D molecules with `rdkit` and `py3Dmol`|
|Checking your work|`np.polyfit` gives the same least-squares line|

## ⚙️ Initialization — DO NOT EDIT
The next two cells load the course tools (auto-checks, quizzes, styling) and record the lab version. **Do not change them.**

👉 **Run the cell below first** (Shift+Enter) before anything else in this lab.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
from init import *

In [ ]:
# 🔖 Lab version — do not edit. Used by the automated checks below.
ver = '2026V108'

### 🧰 Standard imports — start your own notebooks with these
The `init` cell above loads the course tools. The four lines below are the **only** imports you need for your own data science notebooks — copy them into the first cell of any new notebook:

```python
import numpy as np                 # arrays, np.arange, np.random.choice, np.mean, np.std ...
from datascience import *          # Table, make_array, are ...
%matplotlib inline
import matplotlib.pyplot as plt    # plotting
```
Run them here too so every name is defined:

In [ ]:
# Standard imports for this lab
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Now run the cell below to enter your name.

In [ ]:
# Enter your name as a string
name = ...

## 1. How Much Do Sample Means Vary?

In earlier labs we saw that the means of random samples cluster around the population mean, and that **larger samples cluster more tightly**. In this section we put a number on that clustering. The number we use — the **standard deviation (SD)** — will become the ruler for everything else in this lab. See [Ch. 14.5](https://inferentialthinking.com/chapters/14/5/Variability_of_the_Sample_Mean.html).

Our population is the salaries of all employees of the City of San Francisco in 2014. The city reported a mean salary of about \$75,464.

In [ ]:
salaries = Table.read_table('sf_salaries_2014.csv').select('salary')
salaries

In [ ]:
salary_mean = np.mean(salaries.column('salary'))
pop_sd = np.std(salaries.column('salary'))
print(f'Population mean: ${salary_mean:,.2f}')
print(f'Population SD:   ${pop_sd:,.2f}')

In [ ]:
salaries.hist('salary', bins=np.arange(0, 300000 + 10000*2, 10000))
plt.scatter(salary_mean, 0, marker='^', color='red', s=300)
plt.show()

The salaries are clearly **not** normally distributed — they are skewed to the right. Keep that in mind as we go.

### <font color=blue> **Question 1.1.** </font>
Complete the function `simulate_sample_mean`. It draws `repetitions` random samples of size `sample_size` from a column of a table, and returns an array of the sample means. The plotting and printing at the end are done for you.

*Hint:* `np.random.choice(some_array, k)` draws `k` values at random **with replacement** from `some_array`.

In [ ]:
def simulate_sample_mean(table, label, sample_size, repetitions):
    """Empirical distribution of random sample means."""
    population = table.column(label)
    means = make_array()

    for i in np.arange(repetitions):
        new_sample = ...
        new_sample_mean = ...
        means = ...

    # Display the empirical histogram and key numbers — don't change this!
    plt.hist(means, bins=20)
    plt.xlabel('Sample Means')
    plt.title(f'Sample size {sample_size}, {repetitions} repetitions')
    plt.show()
    print(f'Population mean:          {np.mean(population):,.2f}')
    print(f'Average of sample means:  {np.mean(means):,.2f}')
    print(f'Population SD:            {np.std(population):,.2f}')
    print(f'SD of sample means:       {np.std(means):,.2f}')
    return means

In [ ]:
check('tests/q1.1.py')

### <font color=blue> **Question 1.2.** </font>
The first cell below simulates 10,000 sample means for a sample size of 100. Fill in the next two cells to do the same for sample sizes of **400** and **625**. The `plt.xlim` line keeps all three plots on the same axis so you can compare them.

In [ ]:
means_100 = simulate_sample_mean(salaries, 'salary', 100, 10000)

In [ ]:
means_400 = ...

In [ ]:
means_625 = ...

All three histograms are roughly bell-shaped (the Central Limit Theorem at work) even though the salaries themselves are skewed, and all three are centered at the population mean. But they get **narrower** as the sample size grows. How much narrower?

Probability theory predicts:

$$ \text{SD of sample means} = \frac{\text{population SD}}{\sqrt{\text{sample size}}} $$

Fill in `predicted_sd` below using `pop_sd` and the array `sample_sizes`, then compare it with the SDs you observed.

In [ ]:
sample_sizes = make_array(100, 400, 625)
observed_sd = make_array(np.std(means_100), np.std(means_400), np.std(means_625))
predicted_sd = ...

Table().with_columns('Sample size', sample_sizes,
                     'Observed SD of sample means', observed_sd,
                     'Predicted: pop SD / sqrt(n)', predicted_sd)

In [ ]:
check('tests/q1.2.py')

Notice the square root: to make the sample mean **twice** as precise (half the SD), you need **four times** as much data. This is why precise measurements are expensive!

### <font color=blue> **Question 1.3.** </font>
Now keep the sample size fixed at 100 and change only the number of **repetitions** of the simulation. Run the cells below.

In [ ]:
simulate_sample_mean(salaries, 'salary', 100, 500);

In [ ]:
simulate_sample_mean(salaries, 'salary', 100, 5000);

In [ ]:
simulate_sample_mean(salaries, 'salary', 100, 20000);

When I increase the number of **repetitions** of the simulation, keeping the sample size fixed, the SD of the sample means will...

1. Increase
2. Decrease
3. Stay about the same
4. Vary wildly

Assign `repetitions_effect` to the integer corresponding to your answer.

In [ ]:
repetitions_effect = ...

In [ ]:
check('tests/q1.3.py')

### <font color=blue> **Question 1.4.** </font>
How does a single sample compare with the population as the sample gets larger? Run the cells below a few times: each draws one sample and compares its SD with the population SD.

In [ ]:
for n in make_array(10, 200, 1000):
    one_sample = salaries.sample(n, with_replacement=False)
    print(f'n = {n:5d}:  sample SD = {np.std(one_sample.column("salary")):>10,.0f}   (population SD = {pop_sd:,.0f})')

In [ ]:
# Don't change this cell, just run it! Average error in the sample SD for different sample sizes.
sizes = np.arange(10, 200, 10)
sample_sd_errors = make_array()
for n in sizes:
    errors_for_n = make_array()
    for rep in np.arange(100):
        errors_for_n = np.append(errors_for_n, abs(np.std(salaries.sample(n).column('salary')) - pop_sd))
    sample_sd_errors = np.append(sample_sd_errors, np.mean(errors_for_n))
Table().with_columns('Sample size n', sizes, 'Average error in sample SD', sample_sd_errors).plot('Sample size n')
plt.show()

Which of the following is true? Assign `pop_vs_sample` to the integer of your answer.

1. Sample SD gets smaller with increasing sample size; SD of sample means gets smaller with increasing sample size.
2. Sample SD gets larger with increasing sample size; SD of sample means stays the same.
3. Sample SD gets closer to the population SD with increasing sample size; SD of sample means gets smaller with increasing sample size.
4. Sample SD gets closer to the population SD with increasing sample size; SD of sample means stays the same.

In [ ]:
pop_vs_sample = ...

In [ ]:
check('tests/q1.4.py')

### <font color=blue> **Question 1.4 Discussion.** </font>
<font color=green>In your own words, explain the difference between the **SD of a sample** (the spread of the individual salaries) and the **SD of sample means** (the spread of the averages). Which one shrinks as the sample gets larger, and why does that make sense?</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q1.4d.py')

### 1.5 From SD to standard units
The SD is the natural **ruler** for a set of numbers: it tells us how far values typically are from the mean. Instead of saying a salary is "\$150,000", we can say how many SDs it is above average. That is called measuring in **standard units** ([Ch. 14.2](https://inferentialthinking.com/chapters/14/2/Variability.html)):

$$ z = \frac{x - \text{mean}{\text{SD} $$

A value of $z = 2$ means "2 SDs above average"; $z = -0.5$ means "half an SD below average." Any set of numbers converted to standard units has **mean 0 and SD 1** — so two completely different quantities (minutes and minutes, or grams per mole and kelvin) can be put on the **same scale**. That is exactly what we need to compare two variables in Part 2.

### <font color=blue> **Question 1.5.** </font>
Write the function `standard_units`, which converts any array of numbers to standard units. Then use it to find how many SDs a salary of \$150,000 is above the mean, and store it as `z_150k`.

In [ ]:
def standard_units(any_numbers):
    """Convert an array of numbers to standard units."""
    return ...

z_150k = ...
z_150k

In [ ]:
# Check: an array in standard units always has mean 0 and SD 1
salary_su = standard_units(salaries.column('salary'))
print(f'mean = {np.mean(salary_su):.3f},  SD = {np.std(salary_su):.3f}')

In [ ]:
check('tests/q1.5.py')

### 📝 Quick Check — Variability and Standard Units
Test what you've learned about the SD of sample means and standard units. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions08A.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 2. Looking for a Relationship: How Faithful is Old Faithful?

(The clever title comes from [here](http://web.pdx.edu/~jfreder/M212/oldfaithful.pdf).)

Old Faithful is a geyser in Yellowstone National Park, famous for erupting on a fairly regular schedule. Watch it below.

In [ ]:
# The argument to YouTubeVideo is the part of the URL that identifies the video:
#   https://www.youtube.com/watch?v=wE8NDuzt8eg
from IPython.display import YouTubeVideo
YouTubeVideo("wE8NDuzt8eg")

Some eruptions last longer than others. If you visit Yellowstone, you'd like to **predict** when the next eruption will happen so you can explore the park and come back in time. To predict one variable from another, the first step is to understand the **association** between them.

The data set has one row per eruption:
- **duration**: eruption duration, in minutes
- **wait**: time until the *next* eruption, in minutes

In [ ]:
faithful = Table.read_table("faithful-new.csv")
faithful

### Step 1: Look before you calculate
Scientists always **plot the data first**. By convention, the variable we want to **predict** (the *dependent* variable, $y$) goes on the vertical axis, and the variable we predict *from* (the *independent* variable, $x$) goes on the horizontal axis.

There are two ways to make a scatter plot:
```python
faithful.scatter('duration', 'wait')                                  # Table method
plt.scatter(faithful.column('duration'), faithful.column('wait'))     # matplotlib, using arrays
```

### <font color=blue> **Question 2.1.** </font>
Make a scatter plot with `duration` on the horizontal axis and `wait` on the vertical axis. **Do not add any line.**

In [ ]:
...

<font color=green>Describe what you see, using three words scientists use for scatter plots:</font>
- <font color=green>**Direction:** as duration increases, does wait tend to go up, go down, or neither?</font>
- <font color=green>**Form:** do the points follow a straight line, a curve, or separate clusters?</font>
- <font color=green>**Strength:** are the points tightly packed around a trend, or widely scattered?</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q2.1d.py')

### Step 2: Guess the strength of the relationship
The **correlation coefficient** $r$ measures how tightly points cluster around a **straight line**. It is always between $-1$ and $1$:
- $r = 1$: perfect straight line going up; $r = -1$: perfect straight line going down
- $r = 0$: no straight-line trend at all

Before we calculate anything, let's train our eyes. The cell below **simulates** scatter plots with known values of $r$ (see [Ch. 15.1](https://inferentialthinking.com/chapters/15/1/Correlation.html)). Run it and study how the cloud of points changes.

In [ ]:
def r_scatter(r, ax):
    "Draw a simulated scatter plot of 300 points with correlation about r."
    x = np.random.normal(0, 1, 300)
    z = np.random.normal(0, 1, 300)
    y = r * x + np.sqrt(1 - r**2) * z
    ax.scatter(x, y, s=8)
    ax.set_title(f'r = {r}')
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for r_value, ax in zip(make_array(-0.9, -0.5, 0, 0.3, 0.5, 0.7, 0.9, 0.99), axes.flatten()):
    r_scatter(r_value, ax)
plt.tight_layout()
plt.show()

### <font color=blue> **Question 2.2.** </font>
Look back at your Old Faithful scatter plot.
1. Which of $-1$, $0$, or $1$ is closest to the correlation? Assign it to `closest_correlation`.
2. Make a more precise guess using the gallery above and assign it to `r_guess`. We'll see how close you were in Part 3.

In [ ]:
closest_correlation = ...
r_guess = ...

In [ ]:
check('tests/q2.2.py')

### Step 3: Draw a line by eye
Now try to draw the line that best follows the trend. For any line $\hat{y} = \text{slope} \times x + \text{intercept}$, each eruption has an **error** (also called a *residual*):

$$ \text{error} = \text{actual wait} - \text{predicted wait} $$

A good line makes the errors small. Because some errors are positive and some negative, we **square** them before adding them up. The **sum of squared errors (SSE)** measures how badly a line fits — smaller is better ([Ch. 15.3](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html)).

Run the cell below and move the sliders. The vertical gray segments are the errors; try to make the SSE as small as you can.

In [ ]:
from ipywidgets import interact, FloatSlider

def line_errors(x, y, slope, intercept):
    "Actual y minus the y predicted by the line."
    return y - (slope * x + intercept)

def try_a_line(slope, intercept):
    x = faithful.column('duration')
    y = faithful.column('wait')
    sse = np.sum(line_errors(x, y, slope, intercept)**2)
    plt.figure(figsize=(8, 5))
    plt.vlines(x, y, slope * x + intercept, color='gray', alpha=0.4)   # the errors
    plt.scatter(x, y, s=12, label='eruptions')
    xs = make_array(1.5, 5.5)
    plt.plot(xs, slope * xs + intercept, color='red', lw=3, label='your line')
    plt.xlabel('duration (minutes)'); plt.ylabel('wait (minutes)')
    plt.title(f'SSE = {sse:,.0f}')
    plt.legend(); plt.ylim(30, 110)
    plt.show()

interact(try_a_line,
         slope=FloatSlider(min=0, max=20, step=0.25, value=5),
         intercept=FloatSlider(min=0, max=80, step=0.5, value=50));

### <font color=blue> **Question 2.3.** </font>
Record your best slope and intercept from the sliders. Then use the `line_errors` function to compute the **sum of squared errors** of your line and store it as `my_sse`. You'll compare it with the calculated best line in Part 4.

In [ ]:
my_slope = ...
my_intercept = ...
my_errors = line_errors(faithful.column('duration'), faithful.column('wait'), my_slope, my_intercept)
my_sse = ...
print(f'My line: wait = {my_slope} x duration + {my_intercept},   SSE = {my_sse:,.0f}')

In [ ]:
check('tests/q2.3.py')

## 3. Measuring the Relationship with Standard Units

Duration is measured in minutes of eruption, wait in minutes between eruptions, and their spreads are very different. To measure how they move **together**, we first put both on the same ruler: **standard units**, using the `standard_units` function you wrote in Question 1.5.

### <font color=blue> **Question 3.1.** </font>
Create a table called `faithful_standard` with two columns, `"duration (standard units)"` and `"wait (standard units)"`, containing the durations and waits converted to standard units.

In [ ]:
faithful_standard = Table().with_columns(
    "duration (standard units)", ...,
    "wait (standard units)", ...)
faithful_standard

In [ ]:
check('tests/q3.1.py')

### Why standard units reveal correlation
Look at the plot below. Each point is colored by the **product** of its $x$ and $y$ in standard units:
- **Upper-right** (both above average) and **lower-left** (both below average): the product is **positive**.
- **Upper-left** and **lower-right** (one above, one below): the product is **negative**.

If most points fall in the positive quadrants, the variables tend to be above or below average **together** — a positive association.

In [ ]:
xs = faithful_standard.column("duration (standard units)")
ys = faithful_standard.column("wait (standard units)")
products = xs * ys
colors = np.where(products > 0, 'tab:blue', 'tab:orange')

plt.figure(figsize=(6, 6))
plt.scatter(xs, ys, c=colors, s=14)
plt.axhline(0, color='black', lw=1); plt.axvline(0, color='black', lw=1)
plt.xlabel('duration (standard units)'); plt.ylabel('wait (standard units)')
plt.title('Blue: product > 0      Orange: product < 0')
plt.xlim(-2.5, 2.5); plt.ylim(-2.5, 2.5)
plt.show()
print('Points with a positive product:', np.count_nonzero(products > 0), 'of', len(products))

### <font color=blue> **Question 3.2.** </font>
The **correlation coefficient** is the **average of these products** ([Ch. 15.1](https://inferentialthinking.com/chapters/15/1/Correlation.html#calculating-r)):

$$ r = \text{mean of } (x \text{ in standard units}) \times (y \text{ in standard units}) $$

Compute `r` for Old Faithful using `faithful_standard`. How close was your `r_guess`?

In [ ]:
r = ...
print(f'r = {r:.3f}   (your guess was {r_guess})')

In [ ]:
check('tests/q3.2.py')

## 4. Building the Regression Line from *r*

Here is the key fact that connects everything ([Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)):

> **In standard units, the regression line passes through (0, 0) and has slope $r$:**
> $$ \text{wait (su)} = r \times \text{duration (su)} $$

Because $|r| \le 1$, the predicted $y$ is always *closer to average* (in SDs) than $x$ is — an eruption 2 SDs longer than average predicts a wait only $2r$ SDs longer than average. This is the "regression toward the mean" that gave the method its name.

The cell below draws that line through the standard-unit data.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(xs, ys, s=12, label='eruptions')
line_x = make_array(-2, 2)
plt.plot(line_x, r * line_x, color='red', lw=3, label=f'regression line: slope r = {r:.2f}')
plt.plot(line_x, line_x, color='gray', ls='--', label='slope 1 (for comparison)')
plt.xlabel('duration (standard units)'); plt.ylabel('wait (standard units)')
plt.legend(); plt.show()

### Converting back to original units
To go back to minutes, we "stretch" the horizontal axis by $SD_x$ and the vertical axis by $SD_y$. Stretching horizontally makes a line less steep (divide by $SD_x$); stretching vertically makes it steeper (multiply by $SD_y$). So:

$$ \text{slope} = r \cdot \frac{SD_y}{SD_x} $$

The line in standard units passes through $(0, 0)$ — which is the point $(\bar{x}, \bar{y})$ in original units. So:

$$ \text{intercept} = \bar{y} - \text{slope} \times \bar{x} $$

See [Ch. 15.2.5](https://inferentialthinking.com/chapters/15/2/Regression_Line.html#the-equation-of-the-regression-line).

### <font color=blue> **Question 4.1.** </font>
Compute `faithful_slope` and `faithful_intercept` for the regression line in original units (minutes). Use `r`, `np.mean`, and `np.std`.

In [ ]:
duration = faithful.column('duration')
wait = faithful.column('wait')

faithful_slope = ...
faithful_intercept = ...
print(f'Regression line: wait = {faithful_slope:.2f} x duration + {faithful_intercept:.2f}')

In [ ]:
check('tests/q4.1.py')

### Does the line really follow the data? Check with a graph of averages
The regression line is an estimate of the **average** $y$ for each $x$. Let's check that directly: group the eruptions into half-minute bins of duration and average the waits in each bin ([Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)). If a straight line is a good model, those averages (red diamonds) should fall close to the line.

In [ ]:
binned = faithful.with_columns('duration bin', np.round(duration * 2) / 2)
averages = binned.select('duration bin', 'wait').group('duration bin', np.mean)

plt.figure(figsize=(8, 5))
plt.scatter(duration, wait, s=10, alpha=0.4, label='eruptions')
plt.scatter(averages.column('duration bin'), averages.column('wait mean'),
            marker='D', s=70, color='red', label='average wait in each bin')
line_x = make_array(1.5, 5.2)
plt.plot(line_x, faithful_slope * line_x + faithful_intercept, color='black', lw=2, label='your regression line')
plt.xlabel('duration (minutes)'); plt.ylabel('wait (minutes)'); plt.legend()
plt.show()
averages

### <font color=blue> **Question 4.2.** </font>
Compute the sum of squared errors of the regression line and store it as `best_sse`. Compare it with `my_sse` from your hand-drawn line.

In [ ]:
best_sse = ...
print(f'Your line from the sliders: SSE = {my_sse:,.0f}')
print(f'Regression line:            SSE = {best_sse:,.0f}')

In [ ]:
check('tests/q4.2.py')

Unless you got very lucky, the regression line wins. That is no accident: **of all possible straight lines, the regression line has the smallest sum of squared errors.** That is why it is also called the **least-squares line** ([Ch. 15.3](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html)). The formula you built from $r$ and the "best line" you hunted for with sliders are the same thing.

### <font color=blue> **Question 4.3.** </font>
You are a park ranger, and a tourist just watched an eruption that lasted **4 minutes**. Use your regression line to predict the wait until the next eruption, and store it as `wait_after_4`.

In [ ]:
wait_after_4 = ...
wait_after_4

In [ ]:
check('tests/q4.3.py')

<font color=green>When would you tell the tourist to come back so as not to miss the next eruption? Look at the scatter of points around the line at 4 minutes: would you tell them the exact predicted time, or leave a margin? Explain your reasoning.</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q4.3d.py')

![Geezers erupt without warning cartoon](geezers.png)

### <font color=blue> **Question 4.4.** </font> Your regression toolbox
You have now done every step of a regression analysis by hand. Package them as **functions** so you can reuse them on any data set — in Part 5 of this lab, and again in Lab 9. Each function takes a Table `t` and the labels of the $x$ and $y$ columns. Use your `standard_units` function from Question 1.5.

In [ ]:
def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return ...

def slope(t, label_x, label_y):
    "Slope of the regression line predicting label_y from label_x, in original units."
    r = correlation(t, label_x, label_y)
    return ...

def intercept(t, label_x, label_y):
    "Intercept of the regression line predicting label_y from label_x, in original units."
    return ...

In [ ]:
# Test your toolbox: these should match your answers to Questions 3.2 and 4.1
print('r         =', correlation(faithful, 'duration', 'wait'))
print('slope     =', slope(faithful, 'duration', 'wait'))
print('intercept =', intercept(faithful, 'duration', 'wait'))

In [ ]:
check('tests/q4.4.py')

### 📝 Quick Check — Correlation and the Regression Line
Test what you've learned about scatter plots, *r*, and the regression line. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions08B.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 5. Application: Alcohol Structure and Boiling Point

Now apply your toolbox to a chemistry question. **Alcohols** are molecules with an –OH group attached to a carbon chain. What makes some alcohols boil at higher temperatures than others?

To boil, molecules must escape the attractions holding the liquid together. Alcohols have two main kinds of attraction:
- **Hydrogen bonding** between –OH groups — every alcohol here has one –OH, so this is similar across molecules.
- **London dispersion forces** between the carbon chains — these grow with the **size** of the molecule and with how much **surface** two molecules can press against each other.

**Hypothesis:** As molecular weight (MW) increases, boiling point (bp) increases, because larger molecules have stronger dispersion forces.

**Null hypothesis:** MW has no effect on boiling point; any apparent trend is random variation.

Read more: [LibreTexts — Physical Properties of Alcohols](https://chem.libretexts.org/Bookshelves/Organic_Chemistry/Supplemental_Modules_(Organic_Chemistry)/Alcohols/Properties_of_Alcohols/Physical_Properties_of_Alcohols).

### 5.1 Visualizing molecules with RDKit
[RDKit](https://www.rdkit.org/docs/GettingStartedInPython.html) is a free, widely used Python toolkit for chemistry. It reads molecules written as **SMILES** strings — a one-line text code for a structure:

|SMILES|Meaning|Molecule|
|---|---|---|
|`CO`|carbon bonded to oxygen (H's implied)|methanol|
|`CCCCO`|four carbons in a chain, then O|1-butanol|
|`CC(C)CO`|parentheses mark a **branch**|isobutanol|
|`OC1CCCCC1`|matching digits close a **ring**|cyclohexanol|
|`Oc1ccccc1`|lowercase letters are **aromatic** atoms|phenol|

Run the cell below to draw a family of alcohols in 2D, labeled with their boiling points (in kelvin) from our data set.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, Descriptors, Descriptors3D

alcohols = Table().with_columns(
    'name',   make_array('methanol', 'ethanol', '1-propanol', '2-propanol',
                         '1-butanol', 'isobutanol', '2-butanol', 'tert-butanol',
                         '1-pentanol', 'cyclohexanol', 'phenol', '1-octanol'),
    'smiles', make_array('CO', 'CCO', 'CCCO', 'CC(C)O',
                         'CCCCO', 'CC(C)CO', 'CCC(C)O', 'CC(C)(C)O',
                         'CCCCCO', 'OC1CCCCC1', 'Oc1ccccc1', 'CCCCCCCCO'),
    'bp',     make_array(338, 351, 371, 356, 391, 381, 372, 355, 411, 435, 455, 468))

molecules = [Chem.MolFromSmiles(s) for s in alcohols.column('smiles')]
labels = [f'{n}\nbp {b} K' for n, b in zip(alcohols.column('name'), alcohols.column('bp'))]
options = Draw.MolDrawOptions()
options.legendFontSize = 18          # larger labels
options.legendFraction = 0.25        # room for two-line labels
Draw.MolsToGridImage(molecules, molsPerRow=4, subImgSize=(220, 190), legends=labels, drawOptions=options)

RDKit can also **calculate** properties from a structure. For example, `Descriptors.MolWt` computes molecular weight from the atoms:

In [ ]:
alcohols = alcohols.with_columns('MW (RDKit)', [round(Descriptors.MolWt(m), 2) for m in molecules])
alcohols

### 5.2 Molecules in 3D
Real molecules are three-dimensional. Look at the four **butanols** in the table: they have the *same* formula (C₄H₁₀O) and the *same* MW of 74.12 g/mol — they are **isomers** — yet their boiling points range from 355 K to 391 K! MW alone can't explain that. Shape can.

The cell below builds a 3D model of each isomer. RDKit adds the hydrogens, generates 3D coordinates, and relaxes them to a low-energy shape using a force field (MMFF). The [py3Dmol](https://3dmol.csb.pitt.edu/) viewer draws them — **click and drag to rotate, scroll to zoom**.

Try changing `style` to `'sphere'` to see a **space-filling** model, which shows the molecule's outer surface.

In [ ]:
import py3Dmol

def make_3d(smiles):
    "Build a 3D model of a molecule from its SMILES string."
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))    # add hydrogens
    AllChem.EmbedMolecule(mol, randomSeed=42)       # generate 3D coordinates
    AllChem.MMFFOptimizeMolecule(mol)               # relax to a low-energy shape
    return mol

def show_3d(names, smiles_list, style='stick'):
    "Show molecules side by side in rotatable 3D viewers."
    n = len(smiles_list)
    view = py3Dmol.view(width=230 * n, height=260, viewergrid=(1, n))
    for i in np.arange(n):
        mol = make_3d(smiles_list[i])
        view.addModel(Chem.MolToMolBlock(mol), 'mol', viewer=(0, i))
        view.setStyle({style: {'colorscheme': 'Jmol'}}, viewer=(0, i))
        view.addLabel(names[i], {'position': {'x': 0, 'y': -3.5, 'z': 0}, 'fontSize': 12}, viewer=(0, i))
        view.zoomTo(viewer=(0, i))
    view.show()

butanols = alcohols.where('MW (RDKit)', 74.12)
style = 'stick'     # try 'sphere' for a space-filling model
show_3d(butanols.column('name'), butanols.column('smiles'), style)

Notice that 1-butanol is long and stretched out, while tert-butanol is a compact ball. Long molecules can lie alongside each other and touch over more surface, giving stronger dispersion forces.

We can put a number on "stretched out" with the **radius of gyration**: the typical distance of the atoms from the molecule's center. A long, extended molecule has a larger radius of gyration than a compact one of the same mass.

In [ ]:
butanols = butanols.with_columns(
    'radius of gyration (Å)',
    [round(Descriptors3D.RadiusOfGyration(make_3d(s)), 3) for s in butanols.column('smiles')])
butanols.select('name', 'bp', 'radius of gyration (Å)').sort('bp', descending=True)

<font color=green>**Think about it:** For the four butanol isomers, how does boiling point relate to the radius of gyration? Why would a more compact molecule boil at a lower temperature?</font>

Now compare a **chain**, a **ring**, and an **aromatic ring** of similar size. Rotate them and compare their shapes:

In [ ]:
show_3d(make_array('1-hexanol', 'cyclohexanol', 'phenol'),
        make_array('CCCCCCO', 'OC1CCCCC1', 'Oc1ccccc1'), style)

### 5.3 The full data set: boiling point versus MW
`ROH_data.csv` has 72 alcohols with their boiling point `bp` (K), molecular weight `MW` (g/mol), number of `carbons`, and whether they contain a ring (`cyclic`: 1 = yes).

In [ ]:
ROH_data = Table.read_table('ROH_data.csv')
ROH_data

### <font color=blue> **Question 5.1.** </font>
**Look first!** Make a scatter plot of `bp` (vertical) against `MW` (horizontal). No line yet.

In [ ]:
...

<font color=green>Describe the direction, form, and strength of the relationship. Notice the vertical stacks of points at the same MW — what are those, based on Section 5.2? Guess the value of *r*.</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q5.1d.py')

### <font color=blue> **Question 5.2.** </font>
Use your **regression toolbox** from Question 4.4 to compute the correlation, slope, and intercept for predicting `bp` from `MW`.

In [ ]:
roh_r = ...
roh_slope = ...
roh_intercept = ...
print(f'r = {roh_r:.3f}')
print(f'bp = {roh_slope:.3f} x MW + {roh_intercept:.1f}')

In [ ]:
check('tests/q5.2.py')

In [ ]:
# Plot the data colored by structure, with your regression line
chains = ROH_data.where('cyclic', 0)
rings = ROH_data.where('cyclic', 1)
plt.figure(figsize=(8, 5))
plt.scatter(chains.column('MW'), chains.column('bp'), s=25, label='chain alcohols')
plt.scatter(rings.column('MW'), rings.column('bp'), s=25, marker='s', label='ring alcohols')
line_x = make_array(30, 190)
plt.plot(line_x, roh_slope * line_x + roh_intercept, color='black', lw=2, label='regression line')
plt.xlabel('MW [g/mol]'); plt.ylabel('bp [K]'); plt.legend()
plt.show()

### <font color=blue> **Question 5.3.** </font>
The slope means: **change in bp per 1 g/mol increase in MW.** Using your slope, what change in boiling point do you predict for an increase in MW from 100 g/mol to 120 g/mol? Store it as `bp_change` (in K).

In [ ]:
bp_change = ...
bp_change

In [ ]:
check('tests/q5.3.py')

### <font color=blue> **Question 5.4 Discussion.** </font>
<font color=green>
1. In the colored plot, are the ring alcohols mostly above or below the line? Rotate the phenol model in Section 5.2: what feature of its structure might explain this?
2. Within a vertical stack of isomers, which ones fall farthest below the line? Connect your answer to the radius of gyration.
3. Does a strong correlation between MW and bp **prove** that MW causes the boiling point to change? What else changes when MW increases?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q5.4d.py')

#### ⚠️ A warning: *r* only measures **straight-line** association
A correlation near 0 does **not** mean "no relationship." The data below follow a perfect curve, yet *r* is essentially 0 because there is no straight-line trend. This is why we always **look** at the scatter plot first ([Ch. 15.1](https://inferentialthinking.com/chapters/15/1/Correlation.html#properties-of-r)).

In [ ]:
curve = Table().with_columns('x', np.arange(-3, 3.1, 0.25))
curve = curve.with_columns('y', curve.column('x')**2)
curve.scatter('x', 'y')
plt.show()
print('r =', round(correlation(curve, 'x', 'y'), 4))

## 6. Checking Your Work with `np.polyfit`

Scientific software packages have built-in functions for fitting lines. Now that you know exactly what the regression line *is*, you can use them — and understand what they are doing.

`np.polyfit(x, y, deg)` finds the polynomial of degree `deg` that **minimizes the sum of squared errors** — the same least-squares criterion from Question 4.2. It returns the coefficients **starting with the highest power**:

|`deg`|Model|`np.polyfit` returns|
|---|---|---|
|1|$y = m x + b$ (a straight line)|`[m, b]` → slope, then intercept|
|2|$y = a x^2 + b x + c$ (a parabola)|`[a, b, c]`|

For `deg=1`, `np.polyfit` uses calculus (setting the derivative of the SSE to zero) to arrive at exactly the same line you built from $r$. So it should agree with your toolbox. ([NumPy documentation](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html))

### <font color=blue> **Question 6.1.** </font>
Use `np.polyfit` to fit a straight line to the Old Faithful data. Store the slope in `pf_slope` and the intercept in `pf_intercept`. Then use `np.isclose` to check that both match your toolbox, storing `True` or `False` in `matches`.

In [ ]:
coefficients = np.polyfit(..., ..., 1)
pf_slope = coefficients.item(0)
pf_intercept = ...

matches = np.isclose(pf_slope, slope(faithful, 'duration', 'wait')) and np.isclose(pf_intercept, ...)
print(f'np.polyfit:   slope = {pf_slope:.4f}, intercept = {pf_intercept:.4f}')
print(f'Your toolbox: slope = {slope(faithful, "duration", "wait"):.4f}, intercept = {intercept(faithful, "duration", "wait"):.4f}')
print('Match:', matches)

In [ ]:
check('tests/q6.1.py')

From now on, you may use `np.polyfit` as a quick check — but you know what it computes, and you can always build the line yourself from $r$. In **Lab 9** you will use your toolbox to estimate the age of the universe, and use the bootstrap to find out how uncertain that estimate is.

## 📝 Quick Review Quiz
Unlike the quick checks above, this quiz **is recorded** — you get two attempts per question, so take your time.

In [ ]:
# Interactive quiz — run this cell and answer the questions!
from stemds_quiz import open_quiz
quiz_result = open_quiz("questions08C.json", name, user, quiz_id="Lab08C", max_attempts=2)

## 7. Reflection

### <font color=blue> **Question 7.** </font>
At the end of each lab, please include a reflection.
* How did this lab go?
* Are you confident you could fit a line to a new data set using your toolbox?
* How did you find working with molecules in RDKit?
* Were there questions you found especially challenging that you would like your instructor to review in class?
* How long did the lab take you to complete?

Share your feedback so we can continue to improve this class!

**Use the markdown cell below this one and write your reflection on this lab.**

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q7d.py')

## 🏁 Final Step — Check, Submit to Canvas, Then Submit Lab

Follow these three steps **in order**:

1. ✅ **Run the cell below** to test all your answers and see your score.
2. 📤 **Upload to Canvas.** Download your notebook as a `.ipynb` file ([instructions](https://laserchemist.github.io/data/submission-instructions/download-ipynb)) and export an HTML copy ([instructions](https://laserchemist.github.io/data/submission-instructions/export-html)), then upload that HTML file to this lab's Canvas assignment.
3. 🚀 **Only then run the final cell and click Submit Lab.** This records your score to the course gradebook — treat it as confirmation that steps 1 and 2 are already done.

In [ ]:
# Run all tests at once
from gofer.ok import check
correct = 0
questions = ['1.1', '1.2', '1.3', '1.4', '1.4d', '1.5',
             '2.1d', '2.2', '2.3',
             '3.1', '3.2',
             '4.1', '4.2', '4.3', '4.3d', '4.4',
             '5.1d', '5.2', '5.3', '5.4d',
             '6.1',
             '7d']
for x in questions:
    print("Testing question {}: ".format(x))
    display(check("tests/q{}.py".format(x)))
    score = check('tests/q{}.py'.format(str(x)))
    if score.grade == 1.0:
        correct += 1

In [ ]:
from lab_submit import submit_lab
submit_lab(name, user, correct, questions, lab_id="Lab08")